# Edit stability — do in-place edits keep chunks in their clusters?

The known open point from `docs/clustering.md`: every drift experiment added or
removed whole repos; **none ever re-embedded a chunk after a genuine content edit** —
yet in-place edits are the most frequent real-world operation. If ordinary edits shift
embeddings enough to hop clusters or fall below TAU, every commit scatters chunks and
dirties summaries across the tree.

**Design** — replay the UPDATE rule against the real stored state:
- Stored state: v2 chunks at webphone HEAD `c6feb30`, granite-97m embeddings, the 41
  level-1 clusters in `nodes`/`edges` (tree-build).
- Simulated edit batches: webphone files that differ between HEAD and HEAD~3 / HEAD~10 /
  HEAD~15, **M-status only** (pure in-place edits, path unchanged — renames/adds/deletes
  were already covered by steps 20–23). The historical version plays the "edited" version;
  edits are symmetric for this measurement, direction doesn't matter. HEAD~15 includes a
  178-file absolute-imports refactor (`392a556`) — the mass-small-edit stress case.
- Per changed chunk (hash not among the file's stored chunk hashes): embed content, embed
  path (unchanged), α=0.5 combined vector, best similarity against all still-present
  clustered chunks (replaced chunks removed from the pool first, per the UPDATE rule).
  `>= TAU 0.70` joins the best chunk's cluster; below TAU goes to the pending pool.
- **SURVIVED** = joined cluster is one that held this file's chunks before the edit
  (file-level pairing — most files are single-chunk). **HOPPED** = qualified but landed
  in a foreign cluster. **PENDING** = below TAU.


In [ ]:
import subprocess
import tempfile
import time
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path
import sqlite3

import numpy as np
from sentence_transformers import SentenceTransformer

from structuring import split_file_hier

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
REPO = GIT_CACHE / "github.com" / "branow" / "webphone"
PREFIX = "branow/webphone/"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
ALPHA, TAU, BATCH = 0.5, 0.70, 16

db = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)

# stored (pre-edit) state: chunk hashes per file in chunk_no order, level-1 cluster per chunk
chunk_rows = db.execute("SELECT id, path, chunk_no, byte_start, byte_end, hash FROM chunks").fetchall()
node_by_chunk = dict(db.execute(
    "SELECT e.child_id, e.parent_id FROM edges e JOIN nodes n ON n.id = e.parent_id"
    " WHERE n.level = 1 AND e.child_kind = 'chunk'"))

file_hashes = defaultdict(list)   # path -> [hash] in chunk order
file_nodes = defaultdict(set)     # path -> level-1 nodes holding its chunks
node_by_hash = {}                 # clustering is per unique hash -> one node per hash
node_prefix = defaultdict(Counter)
for cid, path, chunk_no, s, e, h in chunk_rows:
    node = node_by_chunk[cid]
    file_hashes[path].append((chunk_no, h))
    file_nodes[path].add(node)
    assert node_by_hash.setdefault(h, node) == node
    node_prefix[node]["/".join(path.split("/")[:4])] += 1
file_hashes = {p: [h for _, h in sorted(v)] for p, v in file_hashes.items()}
node_label = {n: c.most_common(1)[0][0] for n, c in node_prefix.items()}

content_vec = {h: np.frombuffer(v, np.float32) for h, v in db.execute(
    "SELECT hash, vector FROM embeddings WHERE model = ?", (MODEL_NAME,))}
# representative path per unique hash — same GROUP BY selection the stored clustering used
rep_path = dict(db.execute("SELECT hash, path FROM chunks GROUP BY hash"))

_path_vec = {}
def path_vecs(paths):
    todo = sorted({p for p in paths if p not in _path_vec})
    if todo:
        vs = model.encode(todo, batch_size=BATCH, normalize_embeddings=True,
                          show_progress_bar=False)
        for p, v in zip(todo, vs):
            _path_vec[p] = np.asarray(v, np.float32)
    return _path_vec

def combined(cvec, pvec):
    return np.concatenate([np.sqrt(1 - ALPHA) * cvec, np.sqrt(ALPHA) * pvec])

n_webphone = sum(1 for p in file_hashes if p.startswith(PREFIX))
print(f"stored state: {len(chunk_rows)} chunks / {len(node_by_hash)} unique hashes / "
      f"{len(set(node_by_chunk.values()))} level-1 clusters; {n_webphone} webphone files")


## Sanity: the splitter still reproduces the stored chunks

Re-split every webphone file from the cache checkout (= HEAD) and require the exact
stored hash sequence — guards the whole experiment against silent recipe drift.


In [ ]:
t0 = time.monotonic()
mismatches = []
for path, stored in file_hashes.items():
    if not path.startswith(PREFIX):
        continue
    fpath = GIT_CACHE / "github.com" / path
    raw = fpath.read_bytes()
    _, cs = split_file_hier(fpath)
    got = [sha256(raw[c.byte_start:c.byte_end]).hexdigest() for c in cs]
    if got != stored:
        mismatches.append(path)
print(f"{len(mismatches)} mismatching files of {n_webphone} ({time.monotonic() - t0:.1f}s)")
assert not mismatches, mismatches


## The UPDATE-rule replay


In [ ]:
def show_bytes(commit, rel):
    return subprocess.run(["git", "-C", str(REPO), "show", f"{commit}:{rel}"],
                          capture_output=True, check=True).stdout

def split_bytes(rel, body):
    # split_file_hier reads from disk and picks its parser by extension — stage the
    # historical body under the same filename
    tmp = Path(tempfile.mkdtemp()) / Path(rel).name
    tmp.write_bytes(body)
    _, cs = split_file_hier(tmp)
    tmp.unlink(); tmp.parent.rmdir()
    return [(sha256(body[c.byte_start:c.byte_end]).hexdigest(),
             body[c.byte_start:c.byte_end].decode("utf-8", "replace")) for c in cs]

def simulate(base):
    rels = subprocess.run(["git", "-C", str(REPO), "diff", "--name-only", "--diff-filter=M",
                           base, "HEAD"], capture_output=True, text=True, check=True).stdout.split()
    files = [PREFIX + r for r in rels if PREFIX + r in file_hashes]

    # the batch: each modified file's chunks at the "edited" (historical) version
    b_chunks = {f: split_bytes(f.removeprefix(PREFIX), show_bytes(base, f.removeprefix(PREFIX)))
                for f in files}

    # post-edit presence: unmodified files keep their hashes; modified files contribute
    # their edited-version hashes (replaced chunks leave the pool — the UPDATE rule
    # removes them before placing the new versions)
    present = set()
    for path, hs in file_hashes.items():
        if path not in b_chunks:
            present.update(hs)
    for cs in b_chunks.values():
        present.update(h for h, _ in cs)
    pool = [h for h in node_by_hash if h in present]

    # queries: chunks whose hash is new for their file
    queries, n_unchanged = [], 0
    for f, cs in b_chunks.items():
        old = set(file_hashes[f])
        for h, body in cs:
            if h in old:
                n_unchanged += 1
            else:
                queries.append((f, h, body))

    # embed what the corpus has never seen (content-addressed — duplicates are free)
    todo = sorted({h for _, h, _ in queries if h not in content_vec})
    bodies = {h: body for _, h, body in queries}
    for i in range(0, len(todo), BATCH):
        hs = todo[i:i + BATCH]
        vs = model.encode([bodies[h] for h in hs], batch_size=BATCH,
                          normalize_embeddings=True, show_progress_bar=False)
        for h, v in zip(hs, vs):
            content_vec[h] = np.asarray(v, np.float32)

    pv = path_vecs([f for f, _, _ in queries] + [rep_path[h] for h in pool])
    P = np.stack([combined(content_vec[h], pv[rep_path[h]]) for h in pool])

    records = []
    for f, h, body in queries:
        sims = P @ combined(content_vec[h], pv[f])
        j = int(sims.argmax())
        joined, best_sim = node_by_hash[pool[j]], float(sims[j])
        status = ("PENDING" if best_sim < TAU
                  else "SURVIVED" if joined in file_nodes[f] else "HOPPED")
        records.append({"file": f.removeprefix(PREFIX), "path": f, "hash": h,
                        "sim": best_sim, "joined": joined,
                        "old": sorted(file_nodes[f]), "status": status})
    return {"base": base, "files": len(files), "unchanged": n_unchanged,
            "pool": pool, "records": records}


In [ ]:
SPANS = ["HEAD~3", "HEAD~10", "HEAD~15"]
runs = []
for base in SPANS:
    t0 = time.monotonic()
    run = simulate(base)
    runs.append(run)
    rec = run["records"]
    by = Counter(r["status"] for r in rec)
    sims = sorted(r["sim"] for r in rec)
    sha = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", base],
                         capture_output=True, text=True).stdout.strip()
    print(f"== {base} ({sha}) — {run['files']} modified files in corpus, "
          f"{len(rec)} changed chunks + {run['unchanged']} unchanged-hash chunks "
          f"({time.monotonic() - t0:.1f}s)")
    if rec:
        print(f"   SURVIVED {by['SURVIVED']}/{len(rec)} ({by['SURVIVED']/len(rec):.0%})  "
              f"HOPPED {by['HOPPED']}  PENDING {by['PENDING']}  |  "
              f"best-sim min/median/max {sims[0]:.3f}/{sims[len(sims)//2]:.3f}/{sims[-1]:.3f}")
        for r in rec:
            if r["status"] != "SURVIVED":
                old = ", ".join(f"{n}:{node_label[n]}" for n in r["old"])
                print(f"   {r['status']:7} sim={r['sim']:.3f} {r['file']}")
                print(f"           old [{old}] -> joined {r['joined']}:{node_label[r['joined']]}")
    print()


In [ ]:
all_rec = [r for run in runs for r in run["records"]]
by = Counter(r["status"] for r in all_rec)
n = len(all_rec)
print(f"ALL SPANS: {n} changed chunks — "
      f"SURVIVED {by['SURVIVED']} ({by['SURVIVED']/n:.0%}), "
      f"HOPPED {by['HOPPED']} ({by['HOPPED']/n:.0%}), "
      f"PENDING {by['PENDING']} ({by['PENDING']/n:.0%})")
lo = sorted(all_rec, key=lambda r: r["sim"])[:10]
print("\nlowest best-similarities (closest calls):")
for r in lo:
    print(f"  {r['sim']:.3f} {r['status']:8} {r['file']}")


## Control + alternative placement rules

The replay above uses the UPDATE rule as written: join the cluster of the single most
similar chunk. Before blaming the *edits*, test whether that rule can even re-place
**unchanged** content — and whether centroid-based placement (already the misfit
monitor's geometry) does better.


In [ ]:
# CONTROL — is the instability caused by the edits, or by the join rule itself?
# Leave-one-out over ALL stored chunks (no edits at all): drop each chunk from the pool,
# place it by its single nearest remaining chunk (the UPDATE join rule), and check
# whether it lands back in its own cluster.
pv = path_vecs([rep_path[h] for h in node_by_hash])
H = sorted(node_by_hash)
M = np.stack([combined(content_vec[h], pv[rep_path[h]]) for h in H])
S = M @ M.T
np.fill_diagonal(S, -1)
nearest = S.argmax(axis=1)
same = sum(node_by_hash[H[i]] == node_by_hash[H[j]] for i, j in enumerate(nearest))
print(f"leave-one-out nearest-member join, unchanged content: "
      f"{same}/{len(H)} ({same/len(H):.0%}) rejoin their own cluster")

# same control with nearest-CENTROID placement (the misfit monitor's geometry)
nodes_all = sorted(set(node_by_hash.values()))
cent_all = {}
for n in nodes_all:
    idx = [i for i, h in enumerate(H) if node_by_hash[h] == n]
    c = M[idx].mean(axis=0)
    cent_all[n] = c / np.linalg.norm(c)
C = np.stack([cent_all[n] for n in nodes_all])
own = np.array([nodes_all.index(node_by_hash[h]) for h in H])
top = (M @ C.T).argmax(axis=1)
acc = (top == own).mean()
print(f"nearest-centroid placement, unchanged content: {(top == own).sum()}/{len(H)} "
      f"({acc:.0%}) — 1 - misfit rate, by construction")


In [ ]:
# Alternative placement rules on the SAME edit batches:
#   nearest-centroid — place the edited chunk at its most similar cluster centroid
#   sticky           — first check the chunk's own previous cluster(s): if similarity to
#                      that centroid is still >= TAU, stay (the MOVE rule's geometry,
#                      applied to edits); only otherwise re-place by nearest centroid
for run in runs:
    pool = run["pool"]
    by_node = defaultdict(list)
    for h in pool:
        by_node[node_by_hash[h]].append(h)
    pv = path_vecs([rep_path[h] for h in pool] + [r["path"] for r in run["records"]])
    cent, nodes_ = {}, []
    for n, hs in sorted(by_node.items()):
        c = np.stack([combined(content_vec[h], pv[rep_path[h]]) for h in hs]).mean(axis=0)
        cent[n] = c / np.linalg.norm(c)
        nodes_.append(n)
    C = np.stack([cent[n] for n in nodes_])
    n_cent = n_sticky = 0
    for r in run["records"]:
        q = combined(content_vec[r["hash"]], pv[r["path"]])
        s = C @ q
        n_cent += nodes_[int(s.argmax())] in r["old"]
        own_sims = [float(s[nodes_.index(n)]) for n in r["old"] if n in cent]
        n_sticky += bool(own_sims) and max(own_sims) >= TAU
    n = len(run["records"])
    print(f"{run['base']:8} nearest-member {sum(r['status'] == 'SURVIVED' for r in run['records'])}/{n}"
          f"  |  nearest-centroid {n_cent}/{n} ({n_cent/n:.0%})"
          f"  |  sticky-stays {n_sticky}/{n} ({n_sticky/n:.0%})")


## The adopted rule, end-to-end

Full replay of the confirmed rule on the same batches: **sticky re-check first** — the
edited chunk stays if its similarity to its own cluster's **stored (pre-batch) centroid**
is >= TAU. Pre-batch on purpose: new version vs the cluster as it was is exactly the
"did this content drift away?" question, and it survives the edge where a whole cluster's
files are edited in one batch (post-removal that cluster has no members to average — a
first draft used post-removal centroids and all 25 HEAD~15 "escapes" turned out to be
this artifact, own-cluster-empty, not real drift). **Only escapes re-place**, by nearest
post-removal centroid, TAU-gated, else pending. Plus the payoff metric: cluster summaries
dirtied per batch under the old nearest-member rule vs the adopted one (the previous
cluster is always dirty — its content changed; a hop dirties a second one).


In [ ]:
for run in runs:
    pool = run["pool"]
    by_node = defaultdict(list)
    for h in pool:
        by_node[node_by_hash[h]].append(h)
    pv = path_vecs([rep_path[h] for h in pool] + [r["path"] for r in run["records"]])
    nodes_ = sorted(by_node)
    C = np.stack([(lambda c: c / np.linalg.norm(c))(
        np.stack([combined(content_vec[h], pv[rep_path[h]]) for h in by_node[n]]).mean(axis=0))
        for n in nodes_])
    stayed = moved = pending = 0
    dirty_old, dirty_new, moves = set(), set(), []
    stay_sims = []
    for r in run["records"]:
        q = combined(content_vec[r["hash"]], pv[r["path"]])
        own = max(float(cent_all[n] @ q) for n in r["old"])  # stored pre-batch centroids
        dirty_old.update(r["old"]); dirty_old.add(r["joined"])
        dirty_new.update(r["old"])
        if own >= TAU:
            stayed += 1
            stay_sims.append(own)
        else:
            s = C @ q
            j = int(s.argmax())
            if float(s[j]) >= TAU:
                moved += 1; dirty_new.add(nodes_[j])
                moves.append((r["file"], own, float(s[j]), r["old"], nodes_[j]))
            else:
                pending += 1
    n = len(run["records"])
    lo = min(stay_sims) if stay_sims else float("nan")
    print(f"== {run['base']}: STAYED {stayed}/{n} ({stayed/n:.0%}, weakest own-sim {lo:.3f})"
          f"  MOVED {moved}  PENDING {pending}")
    print(f"   dirty cluster summaries: nearest-member rule {len(dirty_old)}, adopted rule {len(dirty_new)}")
    for f, own, best, old, new in moves:
        was = ", ".join(f"{o}:{node_label[o]}" for o in old)
        print(f"   MOVED {f}: own {own:.3f} < TAU -> joins {new}:{node_label[new]} at {best:.3f} (was {was})")
    print()

# control — unchanged content under the adopted rule: sticky re-check alone
own_all = np.array([float(M[i] @ cent_all[node_by_hash[h]]) for i, h in enumerate(H)])
below = int((own_all < TAU).sum())
print(f"control, unchanged chunks: {below}/{len(H)} sit below TAU vs their own centroid "
      f"(only these could ever re-place) -> adopted rule leaves {1 - below/len(H):.0%} untouched in place")


## Observations

**The worry was real but mislocated: in-place edits don't break cluster *fit* — the
original *join rule* was what churned.**

1. **No pending churn, ever.** 216 edited chunks across three batch sizes, zero below
   TAU 0.70 (best-sims 0.848-0.987). In-place edits never orphan a chunk into the
   pending pool.
2. **The nearest-member join rule (cluster of the single most similar chunk) was the
   unstable part.** Edited chunks rejoined their own cluster only 25-68% of the time
   (36% overall) — but the leave-one-out control shows the same rule misplaces 13% of
   *unchanged* chunks too (nearest-centroid: 5%), and every hop lands in a semantically
   adjacent cluster at sim 0.85-0.96, never somewhere insane. The 41 leaf clusters
   finely slice one dense corpus; several centroids sit nearly equidistant from any
   chunk, so a small embedding shift flips the argmax while the chunk's topic didn't
   change. Each gratuitous hop dirties two summaries.
3. **The edited chunk still fits where it was.** Sticky check against the cluster's
   stored *pre-batch* centroid: 100% of edited chunks stay, in every batch, weakest
   own-sim 0.889 — comfortably above TAU. (A first draft checked post-removal centroids
   and produced 25 "escapes" at HEAD~15 — every one was the whole-cluster-edited-in-one-
   batch artifact, an empty own cluster, not real drift. Pre-batch centroids are the
   correct comparison and the final cells use them.)
4. **ADOPTED (2026-07-27) — folded into `docs/clustering.md`'s UPDATE rule.**
   An edited chunk re-checks its own stored centroid first: `>= TAU` stays, only its own
   summary goes dirty; below TAU it re-places via nearest centroid, TAU-gated, else
   pending. Brand-new chunks also place by nearest *centroid* rather than nearest member
   (95% vs 87% leave-one-out self-consistency; O(#clusters) not O(#chunks) at scale).
   Measured payoff: dirty cluster summaries per batch 8->4 / 11->9 / 27->23. Guards:
   over-sticking is the misfit monitor's job (it computes exactly `best_other > own`,
   and LOCAL REPAIRS migrate slow drifters); TAU=0.70 was calibrated on nearest-member
   similarities, centroid sims run lower so refusal fails safe into the pending pool —
   re-measure TAU against centroids with the other constants at the ~50k corpus.
